1. Setup & Load Final Model

In [9]:
import sys
import os
import pandas as pd
import numpy as np

sys.path.append(os.path.abspath(os.path.join('..', 'src')))
from utils import load_data
from features import compute_features
from models import train_model, predict_pass_probabilities

# 1. Load Data
X_train, y_train, X_test = load_data()

# 2. Re-Load your processed training features (to retrain the model on FULL data)
df_train_features = pd.read_csv('../data/processed/train_features.csv')

# 3. Train Final Model on ALL training data (No validation split this time!)
print("Training final model on all 8686 samples...")
X_full = df_train_features.drop(columns=['pass_id', 'target', 'candidate_id', 'sender_id'])
y_full = df_train_features['target']
final_model = train_model(X_full, y_full)

Loading data from c:\Users\ANKRI\Desktop\Q1\ELEN0062-1\Project3\Football-pass-prediction\data\raw...
Training final model on all 8686 samples...
Training Gradient Boosting with Calibration...


2. Process the Test Set

In [10]:
# Check if we already processed the test features to save time
test_feat_path = '../data/processed/test_features.csv'

if os.path.exists(test_feat_path):
    print("Loading pre-computed test features...")
    df_test_features = pd.read_csv(test_feat_path)
else:
    print("Computing features for TEST set (3000 samples)...")
    # We don't have y_test, so pass None
    df_test_features = compute_features(X_test, y_df=None) 
    
    # Save for safety
    df_test_features.to_csv(test_feat_path, index=False)
    print("Saved test features.")

Loading pre-computed test features...


3. Predict on Test Set

In [11]:
print("Predicting probabilities for test set...")
X_test_model = df_test_features.drop(columns=['pass_id', 'target', 'candidate_id', 'sender_id'], errors='ignore')

# Get normalized probabilities
test_probas = predict_pass_probabilities(final_model, X_test_model, df_test_features['pass_id'])

print("Predictions ready.")
print(test_probas.head())

Predicting probabilities for test set...
Predictions ready.
   pass_id  raw_proba  normalized_proba
0        0   0.018014          0.020066
1        0   0.019650          0.021889
2        0   0.004714          0.005251
3        0   0.026731          0.029776
4        0   0.048895          0.054465


4. Format the Submission File

In [12]:
def create_submission_file(probas_df, estimated_acc, filename="submission.csv"):
    """
    Writes the submission file using manual string formatting to ensure
    STRICT COMPLIANCE with Gradescope's expected format (quotes, 'Estimation', etc).
    """
    print(f"Formatting submission... Estimated Acc: {estimated_acc}")
    
    # 1. Pivot the table to get one row per pass_id, columns P_1..P_22
    # We need to ensure we map the probabilities to the correct player columns (P_1 to P_22)
    # Our df_test_features has 'candidate_id', let's use that.
    
    # We need to merge the candidate_ids back if they aren't in probas_df
    # (Assuming probas_df order matches df_test_features order)
    
    # Let's use a pivot table for safety
    # We create a local copy to avoid modifying the original
    df_merged = df_test_features.copy()
    
    # Check if 'normalized_proba' or 'prob' exists
    col_name = 'normalized_proba' if 'normalized_proba' in probas_df.columns else 'prob'
    
    # If probas_df is just the probabilities dataframe, we might need to merge it 
    # But usually test_probas already has 'pass_id' and 'normalized_proba'
    if 'candidate_id' not in probas_df.columns:
        # If probas_df doesn't have candidate_id, we assume it's aligned with df_test_features
        # This is risky. Better to ensure probas_df has candidate_id or merge on index if aligned.
        # Assuming strict alignment from previous steps:
        df_merged['proba'] = probas_df[col_name].values
    else:
        # Safer merge if probas_df has candidate_id
        df_merged = probas_df.copy()
        df_merged.rename(columns={col_name: 'proba'}, inplace=True)

    # Pivot: Index=pass_id, Columns=candidate_id, Values=proba
    pivot = df_merged.pivot(index='pass_id', columns='candidate_id', values='proba')
    
    # Get unique pass IDs sorted
    pass_ids = sorted(pivot.index.unique())

    # Ensure all columns 1..22 exist
    for i in range(1, 23):
        if i not in pivot.columns:
            pivot[i] = 0.0
            
    # Sort columns to be sure P_1, P_2... P_22
    pivot = pivot[range(1, 23)]

    with open(filename, 'w') as f:
        # 1. Write Header (WITH QUOTES)
        # Format: "Id","Predicted","P_1",...
        header_cols = ['"Id"', '"Predicted"'] + [f'"P_{i}"' for i in range(1, 23)]
        f.write(",".join(header_cols) + "\n")
        
        # 2. Write Estimation Line (WITH QUOTES and correct spelling "Estimation")
        # Format: "Estimation",<acc>,0,0,...
        est_line = [f'"Estimation"', str(estimated_acc)] + ["0"] * 22
        f.write(",".join(est_line) + "\n")
        
        # 3. Write Predictions
        for pid in pass_ids:
            row_probs = pivot.loc[pid].values
            
            # Find the player with max probability (Add 1 because index 0 is Player 1)
            predicted_player = np.argmax(row_probs) + 1
            
            # Format the line
            # Format: 0,14,0.01,0.2,...
            line = [str(pid), str(predicted_player)] + [str(p) for p in row_probs]
            f.write(",".join(line) + "\n")
            
    print(f"File '{filename}' created successfully.")

# --- EXECUTE ---
# Use the accuracy you got in notebook 03
# Be realistic! The 'Acc_S' metric depends on this estimate being close to reality.
my_estimated_acc = 0.45  # <--- REPLACE THIS WITH YOUR NOTEBOOK 03 RESULT

# Create the file (using the filename specific to your current attempt)
create_submission_file(test_probas, my_estimated_acc, filename="../submissions/submission_fixed.csv")

Formatting submission... Estimated Acc: 0.45
File '../submissions/submission_fixed.csv' created successfully.
